# Tree Model: CatBoostClassifier con sentence embeddings y metadatos

En este notebook se entrena un modelo `CatBoostClassifier` utilizando sentence embeddings y variables contextuales.

Los embeddings se generan con un modelo preentrenado MiniLM, que transforma cada afirmación en un vector semántico. Estos vectores se combinan con metadatos del dataset, como el tema, el hablante, el cargo, el estado y la afiliación política.

El ajuste de hiperparámetros se realiza mediante Optuna, optimizando parámetros como `depth`, `learning_rate`, `l2_leaf_reg`, `iterations` y `border_count`. Finalmente, se ajusta el threshold para maximizar F1 macro.

In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier, Pool

import optuna
import warnings
warnings.filterwarnings("ignore")

c:\Users\silvi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_prep = pd.read_csv("../data/train_preprocessed.csv")
test_prep = pd.read_csv("../data/test_preprocessed.csv")

print("Train:", train_prep.shape)
print("Test:", test_prep.shape)

display(train_prep.head())
display(test_prep.head())

Train: (8950, 26)
Test: (3836, 25)


,id,label,statement,subject,speaker,speaker_job,state_info,party_affiliation,statement_clean,statement_length,...,speaker_grouped,speaker_grouped_clean,speaker_job_clean,speaker_job_freq,speaker_job_grouped,speaker_job_grouped_clean,state_info_clean,state_info_freq,party_affiliation_clean,party_affiliation_freq
0,81f884c64a7,1,China is in the South China Sea and (building)...,"china,foreign-policy,military",donald-trump,President-Elect,New York,republican,china is in the south china sea and buildinga ...,116,...,donald-trump,donald trump,president elect,247,President-Elect,president elect,new york,579,republican,3947
1,30c2723a188,0,With the resources it takes to execute just ov...,health-care,chris-dodd,U.S. senator,Connecticut,democrat,with the resources it takes to execute just ov...,164,...,chris-dodd,chris dodd,u s senator,236,U.S. senator,u s senator,connecticut,21,democrat,2898
2,6936b216e5d,0,The (Wisconsin) governor has proposed tax give...,"corporations,pundits,taxes,abc-news-week",donna-brazile,Political commentator,"Washington, D.C.",democrat,the wisconsin governor has proposed tax giveaw...,68,...,donna-brazile,donna brazile,political commentator,15,Political commentator,political commentator,washington d c,89,democrat,2898
3,b5cd9195738,1,Says her representation of an ex-boyfriend who...,"candidates-biography,children,ethics,families,...",rebecca-bradley,unknown,unknown,none,says her representation of an exboyfriend who ...,135,...,other,other,unknown,2482,unknown,unknown,unknown,1930,none,1531
4,84f8dac7737,0,At protests in Wisconsin against proposed coll...,"health-care,labor,state-budget",republican-party-wisconsin,unknown,Wisconsin,republican,at protests in wisconsin against proposed coll...,141,...,republican-party-wisconsin,republican party wisconsin,unknown,2482,unknown,unknown,wisconsin,648,republican,3947


,id,statement,subject,speaker,speaker_job,state_info,party_affiliation,statement_clean,statement_length,statement_word_count,...,speaker_grouped,speaker_grouped_clean,speaker_job_clean,speaker_job_freq,speaker_job_grouped,speaker_job_grouped_clean,state_info_clean,state_info_freq,party_affiliation_clean,party_affiliation_freq
0,dc32e5ffa8b,Five members of [the Common Cause Georgia] boa...,"campaign-finance,ethics,government-regulation",kasim-reed,unknown,unknown,democrat,five members of the common cause georgia board...,89,12,...,kasim-reed,kasim reed,unknown,2482.0,unknown,unknown,unknown,1930.0,democrat,2898
1,aa49bb41cab,Theres no negative advertising in my campaign ...,elections,bill-mccollum,unknown,Florida,republican,theres no negative advertising in my campaign ...,53,9,...,bill-mccollum,bill mccollum,unknown,2482.0,unknown,unknown,florida,853.0,republican,3947
2,dddc8d12ac1,Leticia Van de Putte voted to give illegal imm...,"health-care,immigration,public-health",dan-patrick,Lieutenant governor-elect,Texas,republican,leticia van de putte voted to give illegal imm...,141,23,...,dan-patrick,dan patrick,lieutenant governor elect,18.0,Lieutenant governor-elect,lieutenant governor elect,texas,879.0,republican,3947
3,bcfe8f51667,Fiorinas plan would mean slashing Social Secur...,"federal-budget,medicare,social-security",barbara-boxer,U.S. Senator,California,democrat,fiorinas plan would mean slashing social secur...,63,9,...,barbara-boxer,barbara boxer,u s senator,391.0,U.S. Senator,u s senator,california,121.0,democrat,2898
4,eedbbaff5ab,"By the end of his first term, President Obama ...","federal-budget,new-hampshire-2012",mitt-romney,Former governor,Massachusetts,republican,by the end of his first term president obama w...,115,22,...,mitt-romney,mitt romney,former governor,142.0,Former governor,former governor,massachusetts,167.0,republican,3947


In [3]:
text_cols = [
    "statement_clean",
    "subject_clean",
    "speaker_grouped_clean",
    "speaker_job_grouped_clean",
    "state_info_clean",
    "party_affiliation_clean"
]

for col in text_cols:
    train_prep[col] = train_prep[col].fillna("unknown").astype(str)
    test_prep[col] = test_prep[col].fillna("unknown").astype(str)

In [4]:
train_prep["embedding_text"] = (
    train_prep["statement_clean"] + ". " +
    "Subject: " + train_prep["subject_clean"] + ". " +
    "Speaker: " + train_prep["speaker_grouped_clean"] + ". " +
    "Job: " + train_prep["speaker_job_grouped_clean"] + ". " +
    "State: " + train_prep["state_info_clean"] + ". " +
    "Party: " + train_prep["party_affiliation_clean"]
)

test_prep["embedding_text"] = (
    test_prep["statement_clean"] + ". " +
    "Subject: " + test_prep["subject_clean"] + ". " +
    "Speaker: " + test_prep["speaker_grouped_clean"] + ". " +
    "Job: " + test_prep["speaker_job_grouped_clean"] + ". " +
    "State: " + test_prep["state_info_clean"] + ". " +
    "Party: " + test_prep["party_affiliation_clean"]
)

display(train_prep[["embedding_text", "label"]].head())

,embedding_text,label
0,china is in the south china sea and buildinga ...,1
1,with the resources it takes to execute just ov...,0
2,the wisconsin governor has proposed tax giveaw...,0
3,says her representation of an exboyfriend who ...,1
4,at protests in wisconsin against proposed coll...,0


In [5]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2726.38it/s]


In [6]:
train_embeddings = embedding_model.encode(
    train_prep["embedding_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

test_embeddings = embedding_model.encode(
    test_prep["embedding_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Train embeddings:", train_embeddings.shape)
print("Test embeddings:", test_embeddings.shape)

Batches: 100%|██████████| 60/60 [00:14<00:00,  4.27it/s]

Train embeddings: (8950, 384)
Test embeddings: (3836, 384)


In [7]:
np.save("../data/train_embeddings_minilm.npy", train_embeddings)
np.save("../data/test_embeddings_minilm.npy", test_embeddings)

In [8]:
embedding_cols = [f"emb_{i}" for i in range(train_embeddings.shape[1])]

train_emb_df = pd.DataFrame(train_embeddings, columns=embedding_cols)
test_emb_df = pd.DataFrame(test_embeddings, columns=embedding_cols)

print("train_emb_df:", train_emb_df.shape)
print("test_emb_df:", test_emb_df.shape)

train_emb_df: (8950, 384)
test_emb_df: (3836, 384)


In [9]:
cat_cols = [
    "subject_clean",
    "speaker_grouped_clean",
    "speaker_job_grouped_clean",
    "state_info_clean",
    "party_affiliation_clean"
]

for col in cat_cols:
    train_prep[col] = train_prep[col].fillna("unknown").astype(str)
    test_prep[col] = test_prep[col].fillna("unknown").astype(str)

In [10]:
num_cols = [
    "statement_length",
    "statement_word_count",
    "num_subjects"
]

optional_freq_cols = [
    "speaker_freq",
    "speaker_job_freq",
    "state_info_freq",
    "party_affiliation_freq"
]

existing_freq_cols = [
    col for col in optional_freq_cols
    if col in train_prep.columns and col in test_prep.columns
]

num_cols = num_cols + existing_freq_cols

train_prep[num_cols] = train_prep[num_cols].fillna(0)
test_prep[num_cols] = test_prep[num_cols].fillna(0)

print("Variables numéricas:", num_cols)

Variables numéricas: ['statement_length', 'statement_word_count', 'num_subjects', 'speaker_freq', 'speaker_job_freq', 'state_info_freq', 'party_affiliation_freq']


In [11]:
train_meta = train_prep[cat_cols + num_cols].reset_index(drop=True)
test_meta = test_prep[cat_cols + num_cols].reset_index(drop=True)

X = pd.concat(
    [
        train_emb_df.reset_index(drop=True),
        train_meta.reset_index(drop=True)
    ],
    axis=1
)

X_test = pd.concat(
    [
        test_emb_df.reset_index(drop=True),
        test_meta.reset_index(drop=True)
    ],
    axis=1
)

y = train_prep["label"]

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

X: (8950, 396)
y: (8950,)
X_test: (3836, 396)


In [12]:
cat_feature_indices = [X.columns.get_loc(col) for col in cat_cols]

cat_feature_indices

[384, 385, 386, 387, 388]

In [13]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_feature_indices)
test_pool = Pool(X_test, cat_features=cat_feature_indices)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

Train: (7160, 396)
Valid: (1790, 396)


In [14]:
def find_best_threshold_macro(y_true, y_proba):
    thresholds = np.arange(0.40, 0.80, 0.005)
    
    results = []
    
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        
        results.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, y_pred),
            "f1_macro": f1_score(y_true, y_pred, average="macro"),
            "f1_class_1": f1_score(y_true, y_pred)
        })
    
    results_df = pd.DataFrame(results)
    best_row = results_df.sort_values("f1_macro", ascending=False).iloc[0]
    
    return best_row["threshold"], best_row["f1_macro"], results_df

In [15]:
def objective(trial):
    params = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False,
        
        "iterations": trial.suggest_int("iterations", 500, 2500),
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.01, 100.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255)
    }
    
    model = CatBoostClassifier(**params)
    
    model.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=100,
        verbose=False
    )
    
    valid_proba = model.predict_proba(valid_pool)[:, 1]
    
    best_threshold, best_f1_macro, _ = find_best_threshold_macro(
        y_valid,
        valid_proba
    )
    
    trial.set_user_attr("best_threshold", best_threshold)
    
    return best_f1_macro

In [16]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-05-19 11:34:56,334] A new study created in memory with name: no-name-7ac7df6c-6a35-48a4-b141-fbdf6e41a268
Best trial: 0. Best value: 0.629999:   3%|▎         | 1/30 [00:09<04:38,  9.62s/it]

[I 2026-05-19 11:35:05,951] Trial 0 finished with value: 0.6299992244426389 and parameters: {'iterations': 1167, 'depth': 4, 'learning_rate': 0.1100511136177257, 'l2_leaf_reg': 2.790154057769186, 'border_count': 51}. Best is trial 0 with value: 0.6299992244426389.


Best trial: 0. Best value: 0.629999:   7%|▋         | 2/30 [00:25<06:08, 13.16s/it]

[I 2026-05-19 11:35:21,585] Trial 1 finished with value: 0.5723960552142598 and parameters: {'iterations': 2120, 'depth': 9, 'learning_rate': 0.13848674528679447, 'l2_leaf_reg': 0.0683988215505259, 'border_count': 86}. Best is trial 0 with value: 0.6299992244426389.


Best trial: 0. Best value: 0.629999:  10%|█         | 3/30 [05:35<1:06:54, 148.68s/it]

...

[I 2026-05-19 12:54:51,101] Trial 27 finished with value: 0.6281066537209905 and parameters: {'iterations': 2361, 'depth': 8, 'learning_rate': 0.007224613606644958, 'l2_leaf_reg': 0.5087285727650435, 'border_count': 183}. Best is trial 12 with value: 0.6363124151648896.


Best trial: 12. Best value: 0.636312:  97%|█████████▋| 29/30 [1:20:29<05:53, 353.85s/it]

[I 2026-05-19 12:55:25,868] Trial 28 finished with value: 0.63097044464969 and parameters: {'iterations': 1875, 'depth': 6, 'learning_rate': 0.020021505033295278, 'l2_leaf_reg': 4.174444737871938, 'border_count': 108}. Best is trial 12 with value: 0.6363124151648896.


Best trial: 12. Best value: 0.636312: 100%|██████████| 30/30 [1:21:42<00:00, 163.40s/it]

[I 2026-05-19 12:56:38,432] Trial 29 finished with value: 0.6289785749875436 and parameters: {'iterations': 1270, 'depth': 7, 'learning_rate': 0.011766637673998813, 'l2_leaf_reg': 1.789010766875852, 'border_count': 67}. Best is trial 12 with value: 0.6363124151648896.


In [17]:
print("Mejor F1 macro:", study.best_value)
print("Mejores parámetros:")
print(study.best_params)

best_threshold_optuna = study.best_trial.user_attrs["best_threshold"]
print("Mejor threshold Optuna:", best_threshold_optuna)

Mejor F1 macro: 0.6363124151648896
Mejores parámetros:
{'iterations': 2471, 'depth': 7, 'learning_rate': 0.011872985870402789, 'l2_leaf_reg': 6.807277503622012, 'border_count': 239}
Mejor threshold Optuna: 0.6150000000000002


In [18]:
best_params = study.best_params.copy()

best_params.update({
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False
})

best_params

{'iterations': 2471,
 'depth': 7,
 'learning_rate': 0.011872985870402789,
 'l2_leaf_reg': 6.807277503622012,
 'border_count': 239,
 'loss_function': 'Logloss',
 'eval_metric': 'Logloss',
 'random_seed': 42,
 'verbose': False,
 'allow_writing_files': False}

In [19]:
best_model = CatBoostClassifier(**best_params)

best_model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=100,
    verbose=100
)

valid_proba = best_model.predict_proba(valid_pool)[:, 1]

best_threshold, best_f1_macro, threshold_results = find_best_threshold_macro(
    y_valid,
    valid_proba
)

print("Mejor threshold:", best_threshold)
print("Mejor F1 macro:", best_f1_macro)

display(threshold_results.sort_values("f1_macro", ascending=False).head(15))

0:	learn: 0.6914545	test: 0.6916354	best: 0.6916354 (0)	total: 72.1ms	remaining: 2m 58s
100:	learn: 0.6112785	test: 0.6339557	best: 0.6339557 (100)	total: 7.46s	remaining: 2m 55s
200:	learn: 0.5775896	test: 0.6203648	best: 0.6203648 (200)	total: 14.8s	remaining: 2m 47s
300:	learn: 0.5513359	test: 0.6148988	best: 0.6148945 (299)	total: 22.1s	remaining: 2m 39s
400:	learn: 0.5292589	test: 0.6118080	best: 0.6118080 (400)	total: 30.4s	remaining: 2m 36s
500:	learn: 0.5084655	test: 0.6102994	best: 0.6102082 (498)	total: 38.8s	remaining: 2m 32s
600:	learn: 0.4888575	test: 0.6091829	best: 0.6090798 (593)	total: 47.4s	remaining: 2m 27s
700:	learn: 0.4711847	test: 0.6081000	best: 0.6079847 (689)	total: 55.7s	remaining: 2m 20s
800:	learn: 0.4527993	test: 0.6071116	best: 0.6070742 (799)	total: 1m 3s	remaining: 2m 12s
900:	learn: 0.4332598	test: 0.6069798	best: 0.6067835 (841)	total: 1m 12s	remaining: 2m 6s
1000:	learn: 0.4123120	test: 0.6060323	best: 0.6058680 (985)	total: 1m 22s	remaining: 2m 1s
1

,threshold,accuracy,f1_macro,f1_class_1
43,0.615,0.667598,0.636312,0.742981
42,0.610,0.669832,0.635825,0.747112
41,0.605,0.669274,0.631367,0.749577
44,0.620,0.659777,0.631357,0.733712
40,0.600,0.669274,0.628347,0.751678
39,0.595,0.671508,0.628026,0.755204
38,0.590,0.673743,0.627280,0.758877
45,0.625,0.652514,0.626783,0.724779
46,0.630,0.648045,0.625191,0.717742
37,0.585,0.673184,0.621943,0.761127


In [20]:
valid_pred = (valid_proba >= best_threshold).astype(int)

print("Accuracy:", accuracy_score(y_valid, valid_pred))
print("F1 macro:", f1_score(y_valid, valid_pred, average="macro"))
print("F1 clase 1:", f1_score(y_valid, valid_pred))

print(classification_report(y_valid, valid_pred))
print(confusion_matrix(y_valid, valid_pred))

Accuracy: 0.6675977653631285
F1 macro: 0.6363124151648896
F1 clase 1: 0.7429805615550756
              precision    recall  f1-score   support

           0       0.53      0.53      0.53       631
           1       0.74      0.74      0.74      1159

    accuracy                           0.67      1790
   macro avg       0.64      0.64      0.64      1790
weighted avg       0.67      0.67      0.67      1790

[[335 296]
 [299 860]]


In [21]:
full_pool = Pool(X, y, cat_features=cat_feature_indices)

final_model = CatBoostClassifier(**best_params)

final_model.fit(
    full_pool,
    verbose=100
)

test_proba = final_model.predict_proba(test_pool)[:, 1]

0:	learn: 0.6916140	total: 71.1ms	remaining: 2m 55s
100:	learn: 0.6150230	total: 7.91s	remaining: 3m 5s
200:	learn: 0.5844156	total: 19s	remaining: 3m 34s
300:	learn: 0.5619914	total: 30.4s	remaining: 3m 38s
400:	learn: 0.5425366	total: 40.6s	remaining: 3m 29s
500:	learn: 0.5252932	total: 50.9s	remaining: 3m 20s
600:	learn: 0.5086299	total: 1m	remaining: 3m 8s
700:	learn: 0.4926537	total: 1m 10s	remaining: 2m 57s
800:	learn: 0.4758783	total: 1m 20s	remaining: 2m 47s
900:	learn: 0.4581867	total: 1m 30s	remaining: 2m 38s
1000:	learn: 0.4405344	total: 1m 40s	remaining: 2m 27s
1100:	learn: 0.4231324	total: 1m 50s	remaining: 2m 17s
1200:	learn: 0.4068352	total: 1m 59s	remaining: 2m 6s
1300:	learn: 0.3905999	total: 2m 10s	remaining: 1m 57s
1400:	learn: 0.3756570	total: 2m 20s	remaining: 1m 47s
1500:	learn: 0.3610688	total: 2m 30s	remaining: 1m 37s
1600:	learn: 0.3475496	total: 2m 40s	remaining: 1m 27s
1700:	learn: 0.3349955	total: 2m 50s	remaining: 1m 17s
1800:	learn: 0.3226434	total: 2m 59s

In [22]:
thresholds_to_try = [
    round(float(best_threshold), 3),
    0.55,
    0.58,
    0.60,
    0.61,
    0.62,
    0.635,
    0.65
]

thresholds_to_try = sorted(set(thresholds_to_try))

for threshold in thresholds_to_try:
    test_predictions = (test_proba >= threshold).astype(int)
    
    submission = pd.DataFrame({
        "id": test_prep["id"],
        "label": test_predictions
    })
    
    filename = f"submission_catboost_embeddings_threshold_{str(threshold).replace('.', '')}.csv"
    submission.to_csv(filename, index=False)
    
    print("\n==============================")
    print("Archivo:", filename)
    print("Threshold:", threshold)
    print("Distribución:")
    print(submission["label"].value_counts())


Archivo: submission_catboost_embeddings_threshold_055.csv
Threshold: 0.55
Distribución:
label
1    3071
0     765
Name: count, dtype: int64

Archivo: submission_catboost_embeddings_threshold_058.csv
Threshold: 0.58
Distribución:
label
1    2870
0     966
Name: count, dtype: int64

Archivo: submission_catboost_embeddings_threshold_06.csv
Threshold: 0.6
Distribución:
label
1    2717
0    1119
Name: count, dtype: int64

Archivo: submission_catboost_embeddings_threshold_061.csv
Threshold: 0.61
Distribución:
label
1    2622
0    1214
Name: count, dtype: int64

Archivo: submission_catboost_embeddings_threshold_0615.csv
Threshold: 0.615
Distribución:
label
1    2580
0    1256
Name: count, dtype: int64

Archivo: submission_catboost_embeddings_threshold_062.csv
Threshold: 0.62
Distribución:
label
1    2544
0    1292
Name: count, dtype: int64

Archivo: submission_catboost_embeddings_threshold_0635.csv
Threshold: 0.635
Distribución:
label
1    2431
0    1405
Name: count, dtype: int64

Archivo: s

# CatBoost: K-Fold Ensemble

In [23]:
N_SPLITS = 5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)

oof_proba = np.zeros(len(X))
test_proba_folds = np.zeros((len(X_test), N_SPLITS))

fold_scores = []
models = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"\n==============================")
    print(f"Fold {fold + 1} / {N_SPLITS}")
    print(f"==============================")
    
    X_tr = X.iloc[train_idx]
    X_val = X.iloc[valid_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[valid_idx]
    
    train_pool_fold = Pool(X_tr, y_tr, cat_features=cat_feature_indices)
    valid_pool_fold = Pool(X_val, y_val, cat_features=cat_feature_indices)
    test_pool_fold = Pool(X_test, cat_features=cat_feature_indices)
    
    model = CatBoostClassifier(**best_params)
    
    model.fit(
        train_pool_fold,
        eval_set=valid_pool_fold,
        early_stopping_rounds=100,
        verbose=False
    )
    
    val_proba = model.predict_proba(valid_pool_fold)[:, 1]
    test_proba_fold = model.predict_proba(test_pool_fold)[:, 1]
    
    oof_proba[valid_idx] = val_proba
    test_proba_folds[:, fold] = test_proba_fold
    
    val_pred_05 = (val_proba >= 0.5).astype(int)
    
    fold_scores.append({
        "fold": fold + 1,
        "accuracy": accuracy_score(y_val, val_pred_05),
        "f1_macro": f1_score(y_val, val_pred_05, average="macro"),
        "f1_class_1": f1_score(y_val, val_pred_05),
        "best_iteration": model.get_best_iteration()
    })
    
    models.append(model)

fold_scores_df = pd.DataFrame(fold_scores)
display(fold_scores_df)

print("F1 macro medio:", fold_scores_df["f1_macro"].mean())


Fold 1 / 5

Fold 2 / 5

Fold 3 / 5

Fold 4 / 5

Fold 5 / 5


,fold,accuracy,f1_macro,f1_class_1,best_iteration
0,1,0.662570,0.518135,0.781949,1069
1,2,0.660894,0.506593,0.782515,675
2,3,0.671508,0.542701,0.785401,1384
3,4,0.669832,0.536131,0.785169,1473
4,5,0.670391,0.541901,0.784514,1306


F1 macro medio: 0.529092243721973


In [24]:
thresholds = np.arange(0.55, 0.72, 0.005)

threshold_results_oof = []

for threshold in thresholds:
    oof_pred = (oof_proba >= threshold).astype(int)
    
    threshold_results_oof.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y, oof_pred),
        "f1_macro": f1_score(y, oof_pred, average="macro"),
        "f1_class_1": f1_score(y, oof_pred)
    })

threshold_results_oof = pd.DataFrame(threshold_results_oof)

display(threshold_results_oof.sort_values("f1_macro", ascending=False).head(20))

best_threshold_oof = threshold_results_oof.sort_values("f1_macro", ascending=False).iloc[0]["threshold"]

print("Mejor threshold OOF:", best_threshold_oof)

,threshold,accuracy,f1_macro,f1_class_1
14,0.620,0.653520,0.619420,0.733339
13,0.615,0.655196,0.617905,0.737272
15,0.625,0.648827,0.617721,0.726767
12,0.610,0.657207,0.617242,0.740922
16,0.630,0.644916,0.616254,0.721130
11,0.605,0.658659,0.615680,0.744202
17,0.635,0.641899,0.615455,0.716296
19,0.645,0.635419,0.614327,0.704519
20,0.650,0.632179,0.613650,0.698258
18,0.640,0.637207,0.613536,0.709180


Mejor threshold OOF: 0.6200000000000001


In [25]:
test_proba_ensemble = test_proba_folds.mean(axis=1)

thresholds_to_try = [
    0.625,
    0.63,
    0.635,
    0.64,
    0.645,
    0.65,
    round(float(best_threshold_oof), 3)
]

thresholds_to_try = sorted(set(thresholds_to_try))

for threshold in thresholds_to_try:
    test_predictions = (test_proba_ensemble >= threshold).astype(int)
    
    submission = pd.DataFrame({
        "id": test_prep["id"],
        "label": test_predictions
    })
    
    filename = f"submission_catboost_kfold_threshold_{str(threshold).replace('.', '')}.csv"
    submission.to_csv(filename, index=False)
    
    print("\n==============================")
    print("Archivo:", filename)
    print("Threshold:", threshold)
    print("Distribución:")
    print(submission["label"].value_counts())


Archivo: submission_catboost_kfold_threshold_062.csv
Threshold: 0.62
Distribución:
label
1    2501
0    1335
Name: count, dtype: int64

Archivo: submission_catboost_kfold_threshold_0625.csv
Threshold: 0.625
Distribución:
label
1    2440
0    1396
Name: count, dtype: int64

Archivo: submission_catboost_kfold_threshold_063.csv
Threshold: 0.63
Distribución:
label
1    2377
0    1459
Name: count, dtype: int64

Archivo: submission_catboost_kfold_threshold_0635.csv
Threshold: 0.635
Distribución:
label
1    2324
0    1512
Name: count, dtype: int64

Archivo: submission_catboost_kfold_threshold_064.csv
Threshold: 0.64
Distribución:
label
1    2272
0    1564
Name: count, dtype: int64

Archivo: submission_catboost_kfold_threshold_0645.csv
Threshold: 0.645
Distribución:
label
1    2205
0    1631
Name: count, dtype: int64

Archivo: submission_catboost_kfold_threshold_065.csv
Threshold: 0.65
Distribución:
label
1    2148
0    1688
Name: count, dtype: int64


# Modelos adicionales: Transformers
Además del modelo CatBoost con embeddings MiniLM, se probaron modelos Transformer entrenados directamente para la clasificación binaria.

En este caso, el modelo recibe un texto enriquecido que combina la afirmación con metadatos contextuales como el tema, el hablante, el cargo, el estado y el partido político.


# DistilBERT


In [28]:
import sys

!{sys.executable} -m pip install transformers datasets accelerate scikit-learn

     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ---------------------------------------- 82.2/82.2 kB ? eta 0:00:00
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   --------------------------------------- 529.0/529.0 kB 34.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/383.7 kB ? eta -:--:--
   ---------------------------------------- 383.7/383.7 kB ? eta 0:00:00
   ---------------------------------------- 0.0/120.0 kB ? eta -:--:--
   ---------------------------------------- 120.0/120.0 kB ? eta 0:00:00
   ---------------------------------------- 0.0/202.5 kB ? eta -:--:--
   ---------------------------------------- 202.5/202.5 kB ? eta 0:00:00
   ---------------------------------------- 0.0/144.5 kB ? eta -:--:--
   ---------------------------------------- 144.5/144.5 kB ? eta 0:00:00
   ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
   --- ------------------------------------ 2.3/27.3 MB 73


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import random
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [6]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU disponible: False


In [7]:
transformer_text_cols = [
    "statement_clean",
    "subject_clean",
    "speaker_grouped_clean",
    "speaker_job_grouped_clean",
    "state_info_clean",
    "party_affiliation_clean"
]

for col in transformer_text_cols:
    train_prep[col] = train_prep[col].fillna("unknown").astype(str)
    test_prep[col] = test_prep[col].fillna("unknown").astype(str)

train_prep["transformer_text"] = (
    "Statement: " + train_prep["statement_clean"] +
    " Subject: " + train_prep["subject_clean"] +
    " Speaker: " + train_prep["speaker_grouped_clean"] +
    " Job: " + train_prep["speaker_job_grouped_clean"] +
    " State: " + train_prep["state_info_clean"] +
    " Party: " + train_prep["party_affiliation_clean"]
)

test_prep["transformer_text"] = (
    "Statement: " + test_prep["statement_clean"] +
    " Subject: " + test_prep["subject_clean"] +
    " Speaker: " + test_prep["speaker_grouped_clean"] +
    " Job: " + test_prep["speaker_job_grouped_clean"] +
    " State: " + test_prep["state_info_clean"] +
    " Party: " + test_prep["party_affiliation_clean"]
)

display(train_prep[["transformer_text", "label"]].head())

,transformer_text,label
0,Statement: china is in the south china sea and...,1
1,Statement: with the resources it takes to exec...,0
2,Statement: the wisconsin governor has proposed...,0
3,Statement: says her representation of an exboy...,1
4,Statement: at protests in wisconsin against pr...,0


In [8]:
def train_transformer_model(MODEL_NAME, num_epochs=2, batch_size=4, max_length=192):
    
    print("=" * 70)
    print("Entrenando modelo:", MODEL_NAME)
    print("=" * 70)
    
    train_df, valid_df = train_test_split(
        train_prep[["transformer_text", "label"]],
        test_size=0.2,
        random_state=SEED,
        stratify=train_prep["label"]
    )
    
    test_df_transformer = test_prep[["id", "transformer_text"]].copy()
    
    train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
    valid_dataset = Dataset.from_pandas(valid_df.reset_index(drop=True))
    test_dataset = Dataset.from_pandas(test_df_transformer.reset_index(drop=True))
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    def tokenize_function(batch):
        return tokenizer(
            batch["transformer_text"],
            truncation=True,
            max_length=max_length
        )
    
    train_tokenized = train_dataset.map(tokenize_function, batched=True)
    valid_tokenized = valid_dataset.map(tokenize_function, batched=True)
    test_tokenized = test_dataset.map(tokenize_function, batched=True)
    
    train_tokenized = train_tokenized.rename_column("label", "labels")
    valid_tokenized = valid_tokenized.rename_column("label", "labels")
    
    train_tokenized = train_tokenized.remove_columns(["transformer_text"])
    valid_tokenized = valid_tokenized.remove_columns(["transformer_text"])
    test_tokenized = test_tokenized.remove_columns(["transformer_text", "id"])
    
    train_tokenized.set_format("torch")
    valid_tokenized.set_format("torch")
    test_tokenized.set_format("torch")
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro"),
            "f1_class_1": f1_score(labels, preds)
        }
    
    safe_model_name = MODEL_NAME.replace("/", "_")
    
    training_args = TrainingArguments(
        output_dir=f"./transformer_results_{safe_model_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none",
        seed=SEED
    )
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=valid_tokenized,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    eval_results = trainer.evaluate()
    
    print("\nResultados en validación:")
    print(eval_results)
    
    valid_outputs = trainer.predict(valid_tokenized)
    valid_logits = valid_outputs.predictions
    valid_proba = torch.softmax(torch.tensor(valid_logits), dim=1).numpy()[:, 1]
    
    print("\nResultados con threshold 0.5:")
    
    valid_pred_05 = (valid_proba >= 0.5).astype(int)
    
    print("Accuracy:", accuracy_score(valid_df["label"], valid_pred_05))
    print("F1 macro:", f1_score(valid_df["label"], valid_pred_05, average="macro"))
    print("F1 clase 1:", f1_score(valid_df["label"], valid_pred_05))
    print(classification_report(valid_df["label"], valid_pred_05))
    print(confusion_matrix(valid_df["label"], valid_pred_05))
    
    thresholds = np.arange(0.30, 0.80, 0.005)
    
    threshold_results = []
    
    for threshold in thresholds:
        valid_pred = (valid_proba >= threshold).astype(int)
        
        threshold_results.append({
            "threshold": threshold,
            "accuracy": accuracy_score(valid_df["label"], valid_pred),
            "f1_macro": f1_score(valid_df["label"], valid_pred, average="macro"),
            "f1_class_1": f1_score(valid_df["label"], valid_pred)
        })
    
    threshold_results = pd.DataFrame(threshold_results)
    threshold_results = threshold_results.sort_values("f1_macro", ascending=False)
    
    display(threshold_results.head(15))
    
    best_threshold = threshold_results.iloc[0]["threshold"]
    best_f1_macro = threshold_results.iloc[0]["f1_macro"]
    
    print("\nMejor threshold:", best_threshold)
    print("Mejor F1 macro validación:", best_f1_macro)
    
    valid_pred_best = (valid_proba >= best_threshold).astype(int)
    
    print("\nResultados con mejor threshold:")
    print("Accuracy:", accuracy_score(valid_df["label"], valid_pred_best))
    print("F1 macro:", f1_score(valid_df["label"], valid_pred_best, average="macro"))
    print("F1 clase 1:", f1_score(valid_df["label"], valid_pred_best))
    print(classification_report(valid_df["label"], valid_pred_best))
    print(confusion_matrix(valid_df["label"], valid_pred_best))
    
    test_outputs = trainer.predict(test_tokenized)
    test_logits = test_outputs.predictions
    test_proba = torch.softmax(torch.tensor(test_logits), dim=1).numpy()[:, 1]
    
    thresholds_to_try = [
        round(float(best_threshold), 3),
        0.45,
        0.50,
        0.52,
        0.55,
        0.60,
        0.62,
        0.635,
        0.65
    ]
    
    thresholds_to_try = sorted(set(thresholds_to_try))
    
    for threshold in thresholds_to_try:
        test_predictions = (test_proba >= threshold).astype(int)
        
        submission = pd.DataFrame({
            "id": test_prep["id"],
            "label": test_predictions
        })
        
        filename = f"submission_{safe_model_name}_threshold_{str(threshold).replace('.', '')}.csv"
        submission.to_csv(filename, index=False)
        
        print("\n==============================")
        print("Archivo:", filename)
        print("Threshold:", threshold)
        print("Distribución:")
        print(submission["label"].value_counts())
    
    np.save(f"../data/test_proba_{safe_model_name}.npy", test_proba)
    np.save(f"../data/valid_proba_{safe_model_name}.npy", valid_proba)
    
    results = {
        "model_name": MODEL_NAME,
        "best_threshold": best_threshold,
        "best_f1_macro_valid": best_f1_macro,
        "eval_results": eval_results
    }
    
    return results

In [15]:
distilbert_results = train_transformer_model(
    MODEL_NAME="distilbert-base-uncased",
    num_epochs=2,
    batch_size=4,
    max_length=192
)

distilbert_results

Entrenando modelo: distilbert-base-uncased


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9518.23it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Class 1
1,0.629685,0.603320,0.681006,0.595496,0.781477
2,0.492202,0.622248,0.672626,0.629292,0.756037


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.08it/s]


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Class 1
0.492202,0.622248,2,0.672626,0.629292,0.756037


...


Resultados con threshold 0.5:
Accuracy: 0.6726256983240223
F1 macro: 0.629291662720229
F1 clase 1: 0.7560366361365529
              precision    recall  f1-score   support

           0       0.54      0.47      0.50       631
           1       0.73      0.78      0.76      1159

    accuracy                           0.67      1790
   macro avg       0.64      0.63      0.63      1790
weighted avg       0.66      0.67      0.67      1790

[[296 335]
 [251 908]]


,threshold,accuracy,f1_macro,f1_class_1
48,0.540,0.673743,0.637214,0.752332
50,0.550,0.671508,0.636709,0.749147
47,0.535,0.673743,0.636349,0.752961
49,0.545,0.672067,0.636347,0.750319
51,0.555,0.669832,0.635550,0.747328
46,0.530,0.672626,0.634811,0.752325
54,0.570,0.667598,0.634723,0.744306
52,0.560,0.668156,0.634661,0.745283
55,0.575,0.667039,0.634512,0.743546
45,0.525,0.672067,0.634041,0.752007



Mejor threshold: 0.5400000000000003
Mejor F1 macro validación: 0.637213705753579

Resultados con mejor threshold:
Accuracy: 0.6737430167597765
F1 macro: 0.637213705753579
F1 clase 1: 0.7523324851569126
              precision    recall  f1-score   support

           0       0.54      0.51      0.52       631
           1       0.74      0.77      0.75      1159

    accuracy                           0.67      1790
   macro avg       0.64      0.64      0.64      1790
weighted avg       0.67      0.67      0.67      1790

[[319 312]
 [272 887]]



Archivo: submission_distilbert-base-uncased_threshold_045.csv
Threshold: 0.45
Distribución:
label
1    2899
0     937
Name: count, dtype: int64

Archivo: submission_distilbert-base-uncased_threshold_05.csv
Threshold: 0.5
Distribución:
label
1    2762
0    1074
Name: count, dtype: int64

Archivo: submission_distilbert-base-uncased_threshold_052.csv
Threshold: 0.52
Distribución:
label
1    2705
0    1131
Name: count, dtype: int64

Archivo: submission_distilbert-base-uncased_threshold_054.csv
Threshold: 0.54
Distribución:
label
1    2658
0    1178
Name: count, dtype: int64

Archivo: submission_distilbert-base-uncased_threshold_055.csv
Threshold: 0.55
Distribución:
label
1    2630
0    1206
Name: count, dtype: int64

Archivo: submission_distilbert-base-uncased_threshold_06.csv
Threshold: 0.6
Distribución:
label
1    2512
0    1324
Name: count, dtype: int64

Archivo: submission_distilbert-base-uncased_threshold_062.csv
Threshold: 0.62
Distribución:
label
1    2435
0    1401
Name: count, dt

{'model_name': 'distilbert-base-uncased',
 'best_threshold': np.float64(0.5400000000000003),
 'best_f1_macro_valid': np.float64(0.637213705753579),
 'eval_results': {'eval_loss': 0.6222476363182068,
  'eval_accuracy': 0.6726256983240223,
  'eval_f1_macro': 0.629291662720229,
  'eval_f1_class_1': 0.7560366361365529}}

In [14]:
electra_results = train_transformer_model(
    MODEL_NAME="google/electra-small-discriminator",
    num_epochs=2,
    batch_size=4,
    max_length=192
)

electra_results

Entrenando modelo: google/electra-small-discriminator


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11577.15it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized b

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Class 1
1,0.644297,0.622943,0.668156,0.563652,0.777194
2,0.560170,0.627987,0.663128,0.593913,0.761566


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.40it/s]


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Class 1
0.560170,0.627987,2,0.663128,0.593913,0.761566


...


Resultados con threshold 0.5:
Accuracy: 0.6631284916201118
F1 macro: 0.593913270195137
F1 clase 1: 0.7615658362989324
              precision    recall  f1-score   support

           0       0.53      0.35      0.43       631
           1       0.70      0.83      0.76      1159

    accuracy                           0.66      1790
   macro avg       0.62      0.59      0.59      1790
weighted avg       0.64      0.66      0.64      1790

[[224 407]
 [196 963]]


,threshold,accuracy,f1_macro,f1_class_1
82,0.710,0.639665,0.613588,0.713969
80,0.700,0.640782,0.613075,0.716615
81,0.705,0.638547,0.611905,0.713590
83,0.715,0.636872,0.611670,0.710597
78,0.690,0.641341,0.611512,0.719160
79,0.695,0.640223,0.611336,0.717296
73,0.665,0.644134,0.610360,0.725076
75,0.675,0.643017,0.610252,0.723257
72,0.660,0.644134,0.610079,0.725313
77,0.685,0.640782,0.609985,0.719581



Mejor threshold: 0.7100000000000004
Mejor F1 macro validación: 0.6135882525206041

Resultados con mejor threshold:
Accuracy: 0.6396648044692738
F1 macro: 0.6135882525206041
F1 clase 1: 0.7139689578713969
              precision    recall  f1-score   support

           0       0.49      0.54      0.51       631
           1       0.73      0.69      0.71      1159

    accuracy                           0.64      1790
   macro avg       0.61      0.62      0.61      1790
weighted avg       0.65      0.64      0.64      1790

[[340 291]
 [354 805]]



Archivo: submission_google_electra-small-discriminator_threshold_045.csv
Threshold: 0.45
Distribución:
label
1    3119
0     717
Name: count, dtype: int64

Archivo: submission_google_electra-small-discriminator_threshold_05.csv
Threshold: 0.5
Distribución:
label
1    2960
0     876
Name: count, dtype: int64

Archivo: submission_google_electra-small-discriminator_threshold_052.csv
Threshold: 0.52
Distribución:
label
1    2905
0     931
Name: count, dtype: int64

Archivo: submission_google_electra-small-discriminator_threshold_055.csv
Threshold: 0.55
Distribución:
label
1    2831
0    1005
Name: count, dtype: int64

Archivo: submission_google_electra-small-discriminator_threshold_06.csv
Threshold: 0.6
Distribución:
label
1    2727
0    1109
Name: count, dtype: int64

Archivo: submission_google_electra-small-discriminator_threshold_062.csv
Threshold: 0.62
Distribución:
label
1    2679
0    1157
Name: count, dtype: int64

Archivo: submission_google_electra-small-discriminator_threshold_06

{'model_name': 'google/electra-small-discriminator',
 'best_threshold': np.float64(0.7100000000000004),
 'best_f1_macro_valid': np.float64(0.6135882525206041),
 'eval_results': {'eval_loss': 0.6279868483543396,
  'eval_accuracy': 0.6631284916201118,
  'eval_f1_macro': 0.593913270195137,
  'eval_f1_class_1': 0.7615658362989324}}